[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/38_causal_machine_learning/02_potential_outcomes.ipynb)

# Bài 02 — Khung kết quả tiềm năng

**🎯 Mục tiêu**
- Ngôn ngữ **kết quả tiềm năng** (Neyman–Rubin): $Y(1)$, $Y(0)$; hiệu ứng cá nhân (ITE), trung bình (ATE), trên nhóm được điều trị (ATT), có điều kiện (CATE).
- **Vấn đề cơ bản** của suy luận nhân quả: mỗi người chỉ quan sát được một trong hai kết quả.
- Vì sao thí nghiệm ngẫu nhiên giải quyết được; **phân rã thiên lệch chọn lọc** của so sánh ngây thơ trên dữ liệu quan sát (kiểm chứng bằng số).
- **Điều chỉnh bằng hồi quy** (outcome regression): kiểm soát đúng confounder thì gần đúng; thiếu confounder thì sai.
- Khoảng tin cậy bằng bootstrap.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/38_causal_machine_learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

from ml_course.datasets import make_causal_marketing
from ml_course.visualization import set_style

set_style()

## 1. Kết quả tiềm năng

Với mỗi khách hàng $i$ và điều trị nhị phân $T$ (nhận coupon hay không):
- $Y_i(1)$: chi tiêu **nếu** nhận coupon; $Y_i(0)$: chi tiêu **nếu không** nhận.
- **Hiệu ứng cá nhân** (ITE): $\tau_i = Y_i(1) - Y_i(0)$.
- **ATE** $= \mathbb E[Y(1) - Y(0)]$; **ATT** $= \mathbb E[Y(1) - Y(0)\mid T = 1]$ (trên những người thực sự được điều trị); **CATE** $\tau(x) = \mathbb E[Y(1) - Y(0)\mid X = x]$ (theo đặc điểm — bài 04).

**Vấn đề cơ bản** (Holland, 1986): ta chỉ quan sát $Y_i = Y_i(T_i)$ — kết quả **phản thực tế** của mỗi người luôn bị khuyết. Suy luận nhân quả về bản chất là bài toán **dữ liệu khuyết**.

Dữ liệu marketing của chương có cột `true_effect` = $\tau_i$ thật (chỉ có trong dữ liệu mô phỏng), nên ta dựng được **cả hai** kết quả tiềm năng để kiểm tra mọi phương pháp.

In [ ]:
obs = make_causal_marketing(20000, seed=1)                             # dữ liệu QUAN SÁT: coupon không ngẫu nhiên
obs["Y0"] = obs.spend_kvnd - obs.true_effect * obs.coupon              # kết quả tiềm năng (chỉ biết vì là dữ liệu mô phỏng)
obs["Y1"] = obs.Y0 + obs.true_effect
view = obs.head(6)[["age", "income_mvnd", "past_purchases", "coupon", "spend_kvnd"]].copy()
view["Y(1)"] = np.where(obs.head(6).coupon == 1, obs.head(6).Y1.round(1).astype(str), "?")
view["Y(0)"] = np.where(obs.head(6).coupon == 0, obs.head(6).Y0.round(1).astype(str), "?")
print(view.to_string(index=False))
ATE, ATT = obs.true_effect.mean(), obs.true_effect[obs.coupon == 1].mean()
print(f"\nATE thật = {ATE:.1f} nghìn đồng | ATT thật = {ATT:.1f} | tỉ lệ nhận coupon = {obs.coupon.mean():.1%}")

## 2. Thí nghiệm ngẫu nhiên: so sánh hai nhóm là đủ

Nếu phát coupon **ngẫu nhiên** (A/B test), $T$ độc lập với $(Y(0), Y(1))$ → $\mathbb E[Y\mid T = 1] - \mathbb E[Y\mid T = 0] = \mathbb E[Y(1)] - \mathbb E[Y(0)] = \text{ATE}$.

In [ ]:
rct = make_causal_marketing(20000, seed=1, randomized=True)
diff_rct = rct.spend_kvnd[rct.coupon == 1].mean() - rct.spend_kvnd[rct.coupon == 0].mean()
se = np.sqrt(rct.spend_kvnd[rct.coupon == 1].var() / (rct.coupon == 1).sum() + rct.spend_kvnd[rct.coupon == 0].var() / (rct.coupon == 0).sum())
print(f"thí nghiệm ngẫu nhiên: chênh lệch hai nhóm = {diff_rct:.1f} ± {1.96 * se:.1f} (KTC 95%) | ATE thật = {rct.true_effect.mean():.1f}")

## 3. Dữ liệu quan sát: so sánh ngây thơ bị chệch

Trong dữ liệu quan sát, coupon được gửi nhiều hơn cho khách **mua nhiều trước đây** và **thu nhập cao** — những người vốn đã chi tiêu nhiều. Phân rã (với $Y(0)$ dựng được):

$$\underbrace{\mathbb E[Y\mid T{=}1] - \mathbb E[Y\mid T{=}0]}_{\text{so sánh ngây thơ}} = \underbrace{\mathbb E[Y(1) - Y(0)\mid T{=}1]}_{\text{ATT}} + \underbrace{\mathbb E[Y(0)\mid T{=}1] - \mathbb E[Y(0)\mid T{=}0]}_{\text{thiên lệch chọn lọc}}$$

In [ ]:
t1, t0 = obs.coupon == 1, obs.coupon == 0
naive = obs.spend_kvnd[t1].mean() - obs.spend_kvnd[t0].mean()
selection_bias = obs.Y0[t1].mean() - obs.Y0[t0].mean()
print(f"so sánh ngây thơ = {naive:.1f}")
print(f"ATT = {ATT:.1f} | thiên lệch chọn lọc = {selection_bias:.1f} | ATT + thiên lệch = {ATT + selection_bias:.1f}")
print(obs.groupby("coupon")[["income_mvnd", "past_purchases", "age"]].mean().round(2))

Một nửa "hiệu ứng" ngây thơ là **thiên lệch chọn lọc**: người nhận coupon **dù không có coupon** vẫn chi nhiều hơn khoảng 100 nghìn. Bảng trung bình cho thấy hai nhóm khác nhau ở thu nhập và lịch sử mua (confounder) nhưng **giống nhau ở tuổi** (tuổi không ảnh hưởng việc phát coupon).

## 4. Điều chỉnh bằng hồi quy

Dưới giả định **không có confounder chưa đo** ($(Y(0), Y(1)) \perp T\mid X$) và **overlap**, ta ước lượng $\mu_t(x) = \mathbb E[Y\mid T = t, X = x]$ rồi lấy $\widehat{\text{ATE}} = \frac1n\sum_i[\hat\mu_1(x_i) - \hat\mu_0(x_i)]$. Với mô hình tuyến tính $Y \sim T + X$, hệ số của $T$ là ước lượng (khi hiệu ứng không tương tác với $X$ theo cách làm lệch trọng số).

In [ ]:
def ols_effect(columns, data=obs):
    X = data[["coupon"] + columns].values
    return LinearRegression().fit(X, data.spend_kvnd).coef_[0]


rows = [("so sánh ngây thơ", naive),
        ("OLS: coupon + tuổi (thiếu confounder)", ols_effect(["age"])),
        ("OLS: coupon + thu nhập (thiếu lịch sử mua)", ols_effect(["income_mvnd"])),
        ("OLS: coupon + thu nhập + lịch sử mua + tuổi", ols_effect(["income_mvnd", "past_purchases", "age"]))]


def g_computation(data, features):
    """Hồi quy linh hoạt (GBM) cho từng nhóm, rồi trung bình μ1(x) − μ0(x) trên mọi người."""
    X, T, Y = data[features].values, data.coupon.values, data.spend_kvnd.values
    mu1 = HistGradientBoostingRegressor(random_state=0).fit(X[T == 1], Y[T == 1]).predict(X)
    mu0 = HistGradientBoostingRegressor(random_state=0).fit(X[T == 0], Y[T == 0]).predict(X)
    return np.mean(mu1 - mu0)


rows.append(("GBM theo nhóm (g-computation), đủ confounder", g_computation(obs, ["income_mvnd", "past_purchases", "age"])))
print(pd.DataFrame(rows, columns=["phương pháp", "ước lượng ATE"]).round(1).to_string(index=False))
print(f"ATE thật = {ATE:.1f}")

- Thiếu confounder (chỉ có tuổi) → vẫn chệch y như ngây thơ; thiếu một trong hai confounder → chệch một phần.
- Kiểm soát **đủ** confounder → gần ATE thật, với cả mô hình tuyến tính lẫn GBM.
- Giả định "đủ confounder" **không kiểm chứng được từ dữ liệu**: phải dựa vào hiểu biết về **cơ chế** phát coupon (ai quyết định, dựa trên thông tin gì). Nếu quy tắc phát coupon dùng thông tin không có trong dữ liệu (vd đánh giá cảm tính của nhân viên bán hàng), mọi phương pháp quan sát đều có thể chệch.

## 5. Bất định: bootstrap khoảng tin cậy

In [ ]:
rng = np.random.default_rng(0)
boot = []
for _ in range(200):
    sample = obs.sample(len(obs), replace=True, random_state=int(rng.integers(1_000_000_000)))
    boot.append(ols_effect(["income_mvnd", "past_purchases", "age"], sample))
lo, hi = np.quantile(boot, [0.025, 0.975])
print(f"OLS điều chỉnh: {rows[3][1]:.1f}, KTC 95% bootstrap = [{lo:.1f}, {hi:.1f}] | ATE thật = {ATE:.1f}")
plt.figure(figsize=(6, 3))
plt.hist(boot, bins=30)
plt.axvline(ATE, color="k", ls=":", label="ATE thật")
plt.axvline(naive, color="C3", ls="--", label="ngây thơ")
plt.legend()
plt.tight_layout()
plt.show()

Khoảng tin cậy chỉ phản ánh **bất định do mẫu**, không phản ánh sai lệch do confounder chưa đo — nếu giả định sai, khoảng tin cậy hẹp vẫn có thể nằm xa giá trị thật. Phân tích **độ nhạy** (sensitivity analysis: "confounder chưa đo phải mạnh đến đâu để đảo kết luận?") là bước nên làm khi báo cáo kết quả quan sát.

## 🧠 Kiểm tra nhanh

1. Định nghĩa ITE, ATE, ATT, CATE.
2. "Vấn đề cơ bản của suy luận nhân quả" là gì?
3. Vì sao thí nghiệm ngẫu nhiên làm so sánh hai nhóm không chệch?
4. Phân rã so sánh ngây thơ thành ATT + thiên lệch chọn lọc. Thiên lệch chọn lọc trong ví dụ đến từ đâu?
5. Điều chỉnh bằng hồi quy cần những giả định nào? Giả định nào không kiểm chứng được?

## 📌 Tóm tắt

- Kết quả tiềm năng $Y(1), Y(0)$; chỉ quan sát được một → suy luận nhân quả là bài toán dữ liệu khuyết.
- RCT: so sánh nhóm = ATE. Quan sát: so sánh ngây thơ = ATT + thiên lệch chọn lọc.
- Điều chỉnh (OLS, g-computation) đúng khi đủ confounder + overlap; thiếu confounder thì chệch; khoảng tin cậy không cứu được giả định sai.

## 📚 Tài liệu tham khảo

- Rubin (1974) — *Estimating causal effects of treatments in randomized and nonrandomized studies*
- Holland (1986) — *Statistics and Causal Inference*
- Imbens & Rubin (2015) — *Causal Inference for Statistics, Social, and Biomedical Sciences*
- Angrist & Pischke (2009) — *Mostly Harmless Econometrics*
- Hernán & Robins (2020) — *Causal Inference: What If*, phần I